|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 6. The engine is a server, and the server has metrics. Now
the failures come from the layers around the engine: the event loop, the
network, the proxy, and the dashboard. Several tickets in this file are about
a number that is wrong, not about an engine that is wrong.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 16. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 6.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| Metric | Definition |
|---|---|
| TTFT | from the arrival of the request to its first token |
| TPOT | the average time between the output tokens of one request |
| ITL | each single time between two output tokens |
| goodput | the requests each second that meet their promise |

- Little's law: requests in the system = arrival rate x time in the system.
- A percentile of a union is not the average of the percentiles of the
  parts.
- Server-Sent Events (SSE) send each token as one small chunk of an HTTP
  response that stays open.

# Ticket 1: every stream freezes at the same moment

**Severity:** medium. **Reported by:** users.

> Several times each hour, all the streams stop for about half a second,
> and then continue.

**Evidence**

- The freezes appear in the client logs as a gap of 400 to 450 ms between
  two tokens, at the same wall-clock moment for every stream.
- The engine log shows a step time of 22 ms, flat, including during the
  freezes.
- The request handler:

  ```python
  @app.post('/v1/completions')
  async def completions(req: CompletionRequest):
      ids = tokenizer.encode(req.prompt)
      return StreamingResponse(engine.generate(ids, req.params))
  ```

- The freezes start when a user sends a document. Chunked prefill is on.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: the KV cache is full at night

**Severity:** medium. **Reported by:** the capacity team.

> The KV cache is 95% full even at night, when few users are active. We
> need more GPU memory.

**Evidence**

- 30% of the users press "stop" in the app, on average after 50 tokens.
  They stop the answers that ramble.
- The answers that ramble would continue to `max_tokens = 4096`. The
  other answers end naturally after about 400 tokens.
- The metrics for one hour: 12.0 million tokens generated, 2.35 million
  tokens delivered to the clients.
- The stream handler catches no exception when the client goes away.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: the dashboard says 80 ms, the users say 3 seconds

**Severity:** medium. **Reported by:** the product team.

> The TTFT on the dashboard is 80 ms at the p50. Users say that they wait
> about 3 seconds for the first word.

**Evidence**

- The code that starts the TTFT clock:

  ```python
  def admit(self, seq):
      seq.t_start = time.monotonic()
      self.running.append(seq)
  ```

- The queue wait metric, p50: 2.9 s.
- The users are on mobile networks. The team thinks that the network is
  slow.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: a p99 of 480 ms that nobody sees

**Severity:** medium. **Reported by:** the SRE team.

> The fleet p99 latency on the dashboard is 480 ms. The slowest 1% of
> the requests in the client logs take more than 2 seconds.

**Evidence**

- The fleet has 10 pods. Each pod reports its own p99.
- The dashboard computes the fleet p99 as the average of the 10 values.
- 9 pods report a p99 of 200 ms. One pod reports 3,000 ms. That pod has a
  bad fan, and its GPU throttles.
- Each pod serves 10% of the traffic.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: streaming that arrives all at once

**Severity:** high. **Reported by:** the front-end team.

> In production, the text does not stream. The whole answer appears at
> the end. On a laptop against the pod, it streams well.

**Evidence**

- `curl` to the pod directly: the first chunk after 150 ms, then a chunk
  every 25 ms.
- `curl` through the public URL: all chunks arrive at the same moment,
  after 9.8 s.
- A long answer takes 9.8 s to generate.
- The public URL goes through an nginx ingress. Its config has no setting
  about buffering.
- The last release changed the JSON format of each chunk.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: the release that made us six times faster

**Severity:** low. It goes into a press release. **Reported by:** the
marketing team.

> Release 2.3 raised the throughput from 1,900 to 11,400 tokens/s. We
> want to announce a 6x speedup.

**Evidence**

- The release notes contain: "unify the token counting in the metrics".
- The request rate and the GPU step times did not change.
- The average request has 1,500 prompt tokens and 300 output tokens.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: production gets a third of the load test

**Severity:** medium. **Reported by:** the performance team.

> The load test with 50 users gives 1,000 tokens/s. Production has 50
> users and gives 310 tokens/s. Something in production is slow.

**Evidence**

- In the load test, each virtual user sends the next request as soon as
  the answer ends.
- In production, a user reads the answer and thinks for about 20 s
  before the next request.
- A request takes about 10 s from arrival to the last token, in both
  cases.
- The GPU utilization in production is about 35%.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**